# YOLOv8 — Vehicle + Plate Detection (Colab)

One multi-class detector (`car, motorbike, truck, bus, plate`) trained on a Colab **GPU** runtime, driven from VS Code via the Colab extension.

**Trigger:** Select Kernel → a Colab **GPU** runtime → Run All. TPU will not work (Ultralytics = CUDA).

**Resumable:** checkpoints land on Drive every epoch. After a disconnect, run the **Resume** cell (6b) instead of re-training.

Deliverable: `best.pt` + ONNX in Drive `weights/`; metrics row appended to `src/ml/experiments.csv`.

## 0. Config — edit here, then Run All

In [ ]:
# ── Config ───────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/UIT2026-DoAnCuoiKi'
DATASET_ZIP = f'{DRIVE_ROOT}/datasets/parking-detect-v1.zip'  # zipped dataset on Drive
DATASET_DIR = '/content/dataset'                             # local unzip target (fast reads)
RUN_NAME    = 'yolov8n-detect-v1'

MODEL   = 'yolov8n.pt'   # edge target: start n/s, scale up only with evidence
IMGSZ   = 640
EPOCHS  = 100
BATCH   = 16
CLASSES = ['car', 'motorbike', 'truck', 'bus', 'plate']  # one multi-class model

CKPT_DIR    = f'{DRIVE_ROOT}/checkpoints'   # ultralytics writes last.pt/best.pt here every epoch
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'

## 1. Setup — pinned installs

In [ ]:
!pip install -q ultralytics==8.3.0  # pinned — bump deliberately, never float
import ultralytics
ultralytics.checks()

## 2. Runtime check — must be GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No CUDA GPU. In VS Code: Select Kernel → a Colab GPU runtime (not TPU/CPU), then Run All.'
)
print('GPU:', torch.cuda.get_device_name(0))

## 3. Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
for d in (CKPT_DIR, WEIGHTS_DIR):
    os.makedirs(d, exist_ok=True)
print('Drive mounted. checkpoints →', CKPT_DIR)

## 4. Stage dataset — Drive → local `/content` (Drive-direct reads are slow)

In [ ]:
import os, zipfile, glob

os.makedirs(DATASET_DIR, exist_ok=True)
assert os.path.exists(DATASET_ZIP), f'Dataset zip not found on Drive: {DATASET_ZIP}'
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(DATASET_DIR)

# verify counts against expected layout: images/<split>, labels/<split>
for split in ('train', 'val'):
    imgs = glob.glob(f'{DATASET_DIR}/images/{split}/*')
    lbls = glob.glob(f'{DATASET_DIR}/labels/{split}/*.txt')
    print(f'{split}: {len(imgs)} images, {len(lbls)} labels')
    assert len(imgs) > 0, f'No images in {split} — check zip layout'

## 5. Dataset YAML — inlined (local repo files are not on the runtime)

In [ ]:
import yaml

data_yaml = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {i: c for i, c in enumerate(CLASSES)},
}
DATA_YAML_PATH = '/content/parking-detect.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(open(DATA_YAML_PATH).read())

## 6. Train — checkpoints to Drive every epoch

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML_PATH,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=CKPT_DIR,   # → Drive; last.pt updated every epoch (resume-safe)
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    degrees=10.0,       # small rotation — plates shot at gate angles
    seed=7,
)

## 6b. Resume — run ONLY after a disconnect (skip on a clean run)

In [ ]:
# Uncomment + run if training was interrupted. Resumes from Drive last.pt.
# from ultralytics import YOLO
# model = YOLO(f'{CKPT_DIR}/{RUN_NAME}/weights/last.pt')
# model.train(resume=True)

## 7. Validate — per-class metrics

In [ ]:
from ultralytics import YOLO

best = f'{CKPT_DIR}/{RUN_NAME}/weights/best.pt'
metrics = YOLO(best).val(data=DATA_YAML_PATH, imgsz=IMGSZ, split='val')
print('mAP50    :', round(metrics.box.map50, 4))
print('mAP50-95 :', round(metrics.box.map, 4))
print('precision:', round(metrics.box.mp, 4))
print('recall   :', round(metrics.box.mr, 4))

## 8. Export ONNX + handoff

In [ ]:
import shutil, datetime, os
from ultralytics import YOLO

best = f'{CKPT_DIR}/{RUN_NAME}/weights/best.pt'
onnx_path = YOLO(best).export(format='onnx', opset=17, imgsz=IMGSZ, dynamic=False)

os.makedirs(WEIGHTS_DIR, exist_ok=True)
shutil.copy(best, f'{WEIGHTS_DIR}/{RUN_NAME}.pt')
shutil.copy(onnx_path, f'{WEIGHTS_DIR}/{RUN_NAME}.onnx')

# experiments.csv row — runtime cannot write the local repo, so print + paste into src/ml/experiments.csv
row = ','.join(str(x) for x in [
    datetime.date.today().isoformat(), MODEL, os.path.basename(DATASET_ZIP),
    f'imgsz={IMGSZ};epochs={EPOCHS};batch={BATCH}',
    round(metrics.box.map50, 4), round(metrics.box.map, 4),
    round(metrics.box.mp, 4), round(metrics.box.mr, 4),
    f'{WEIGHTS_DIR}/{RUN_NAME}.pt',
])
print('append to src/ml/experiments.csv:')
print(row)